In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
import time, glob
import numpy as np
import pynq
from pynq import Overlay, allocate, Clocks
from tqdm import tqdm
np.set_printoptions(linewidth=120, edgeitems=4, suppress=True)

In [2]:
# Load the overlay (tiled_conv.hwh must be in the same folder as the .bit)
bitstream = "tiled_conv.bit"
overlay = Overlay(bitstream)
overlay.download()
Clocks.fclk0_mhz = 100.0            # set the PL clock to 100 MHz (the board can start at a different value)
print("PL clock (MHz):", Clocks.fclk0_mhz)

dut = overlay.tiled_conv_0
print(dut.register_map)

PL clock (MHz): 100.0
RegisterMap {
  CTRL = Register(AP_START=0, AP_DONE=0, AP_IDLE=1, AP_READY=0, RESERVED_1=0, AUTO_RESTART=0, RESERVED_2=0, INTERRUPT=0, RESERVED_3=0),
  GIER = Register(Enable=0, RESERVED=0),
  IP_IER = Register(CHAN0_INT_EN=0, CHAN1_INT_EN=0, RESERVED_0=0),
  IP_ISR = Register(CHAN0_INT_ST=0, CHAN1_INT_ST=0, RESERVED_0=0),
  input_fm_1 = Register(input_fm=write-only),
  input_fm_2 = Register(input_fm=write-only),
  weights_1 = Register(weights=write-only),
  weights_2 = Register(weights=write-only),
  biases_1 = Register(biases=write-only),
  biases_2 = Register(biases=write-only),
  output_fm_1 = Register(output_fm=write-only),
  output_fm_2 = Register(output_fm=write-only)
}


In [3]:
# Layer and tiling parameters (golden design: 14x14 tile, 16 kernels per group, ap_fixed<8,4>)
M, I, K, S, P, N, O = 3, 224, 3, 1, 1, 64, 224
TILE_HW = 14
NUM_TILE_RC = I // TILE_HW
MARGIN = 1
IN_BUF_DEPTH = 3
IN_BUF_HW = TILE_HW + 2 * MARGIN
OUT_BUF_DEPTH = 16
OUT_BUF_HW = TILE_HW
NUM_KG = N // OUT_BUF_DEPTH

FM_W, FM_I = 8, 4
SCALE = 1 << (FM_W - FM_I)           # real value = stored integer / SCALE

In [4]:
# Load the data files (or random data if they are not found) and convert to ap_fixed<8,4>
def find_bin(nbytes):
    for pat in ["./*.bin", "./params/*.bin", "../params/*.bin", "./*/*.bin"]:
        for f in sorted(glob.glob(pat)):
            if os.path.getsize(f) == nbytes:
                return f
    return None

f_in, f_wt, f_b, f_out = find_bin(M*I*I*4), find_bin(N*M*K*K*4), find_bin(N*4), find_bin(N*O*O*4)
print("files:", f_in, f_wt, f_b, f_out)
if f_in and f_wt and f_b:
    x_f = np.fromfile(f_in, dtype=np.float32).reshape(M, I, I)
    w_f = np.fromfile(f_wt, dtype=np.float32).reshape(N, M, K, K)
    b_f = np.fromfile(f_b,  dtype=np.float32)
else:
    print("Data files not found: using random data")
    rng = np.random.default_rng(0)
    x_f = rng.normal(0, 1, (M, I, I)).astype(np.float32)
    w_f = rng.uniform(-0.3, 0.3, (N, M, K, K)).astype(np.float32)
    b_f = rng.uniform(-0.1, 0.1, N).astype(np.float32)
golden = np.fromfile(f_out, dtype=np.float32).reshape(N, O, O) if f_out else None

def float_to_fixed(a):
    # float -> ap_fixed<8,4>: drop extra fractional bits (floor), keep 1 signed byte
    return np.clip(np.floor(a.astype(np.float64) * SCALE), -128, 127).astype(np.int8)

input_fm  = allocate((M, I, I),    dtype=np.int8)
weights   = allocate((N, M, K, K), dtype=np.int8)
biases    = allocate((N,),         dtype=np.int8)
output_fm = allocate((N, O, O),    dtype=np.int8)

input_fm[:] = float_to_fixed(x_f)
weights[:]  = float_to_fixed(w_f)
biases[:]   = float_to_fixed(b_f)
output_fm.fill(0)
for b in (input_fm, weights, biases, output_fm):
    b.flush()

files: ../params/input_224.bin ../params/conv1_weights.bin ./params/conv_biases.bin ../params/golden_conv1.bin


In [5]:
# Give the IP the buffer addresses
dut.register_map.input_fm_1  = input_fm.physical_address & 0xFFFFFFFF
dut.register_map.input_fm_2  = input_fm.physical_address >> 32
dut.register_map.weights_1   = weights.physical_address & 0xFFFFFFFF
dut.register_map.weights_2   = weights.physical_address >> 32
dut.register_map.biases_1    = biases.physical_address & 0xFFFFFFFF
dut.register_map.biases_2    = biases.physical_address >> 32
dut.register_map.output_fm_1 = output_fm.physical_address & 0xFFFFFFFF
dut.register_map.output_fm_2 = output_fm.physical_address >> 32

In [6]:
# Run the FPGA IP and time it (1 warm-up run, then the median of 5 runs)
def run_hw():
    output_fm.fill(0)
    output_fm.flush()
    t0 = time.perf_counter()
    dut.write(0x00, 1)                      # ap_start
    while (dut.read(0x00) & 0x2) == 0:      # wait for ap_done
        pass
    t1 = time.perf_counter()
    output_fm.invalidate()
    return (t1 - t0) * 1e3

run_hw()
hw_times = [run_hw() for _ in range(5)]
hw_ms = float(np.median(hw_times))
print("FPGA run times (ms):", [f"{t:.3f}" for t in hw_times])
print(f"FPGA Execution time: {hw_ms:.3f} ms")

FPGA run times (ms): ['93.195', '93.175', '93.167', '93.177', '93.162']
FPGA Execution time: 93.175 ms


In [7]:
# Software (CPU) version: the same tiled convolution in plain Python loops
# It uses the 8-bit values converted back to real numbers (exact in float64).
x_r = input_fm.astype(np.float64) / SCALE
w_r = weights.astype(np.float64) / SCALE
b_r = biases.astype(np.float64) / SCALE

def load_input_tile_from_DRAM_sw(in_fm, in_fm_buf, tr, tc):
    for i in range(IN_BUF_DEPTH):
        for r in range(IN_BUF_HW):
            for c in range(IN_BUF_HW):
                global_r = tr * TILE_HW + r - MARGIN
                global_c = tc * TILE_HW + c - MARGIN
                if 0 <= global_r < I and 0 <= global_c < I:
                    in_fm_buf[i, r, c] = in_fm[i, global_r, global_c]
                else:
                    in_fm_buf[i, r, c] = 0          # zero padding

def load_layer_params_from_DRAM_sw(weights, biases, weights_buf, biases_buf, kernel_group):
    for n in range(OUT_BUF_DEPTH):
        biases_buf[n] = biases[kernel_group * OUT_BUF_DEPTH + n]
        for m in range(IN_BUF_DEPTH):
            for kr in range(K):
                for kc in range(K):
                    weights_buf[n, m, kr, kc] = weights[kernel_group * OUT_BUF_DEPTH + n, m, kr, kc]

def conv_3x3_sw(X_buf, W_buf, B_buf, Y_buf):
    for n in range(OUT_BUF_DEPTH):
        for r in range(OUT_BUF_HW):
            for c in range(OUT_BUF_HW):
                sum = 0
                for m in range(IN_BUF_DEPTH):
                    for kr in range(K):
                        for kc in range(K):
                            sum += ((X_buf[m, r + kr, c + kc] * W_buf[n, m, kr, kc] * SCALE) // 1) / SCALE   # product cut to 4 fractional bits, like the HLS code
                Y_buf[n, r, c] = max(sum + B_buf[n], 0)       # bias + ReLU

def store_output_tile_to_DRAM_sw(out_fm_buf, out_fm, tr, tc, kernel_group):
    for n in range(OUT_BUF_DEPTH):
        for r in range(OUT_BUF_HW):
            for c in range(OUT_BUF_HW):
                out_fm[kernel_group * OUT_BUF_DEPTH + n, tr * TILE_HW + r, tc * TILE_HW + c] = out_fm_buf[n, r, c]

def tiled_conv_sw(input_fm, weights, biases, output_fm):
    conv_in_buf   = np.zeros((IN_BUF_DEPTH, IN_BUF_HW, IN_BUF_HW))
    conv_wt_buf   = np.zeros((OUT_BUF_DEPTH, IN_BUF_DEPTH, K, K))
    conv_bias_buf = np.zeros((OUT_BUF_DEPTH))
    conv_out_buf  = np.zeros((OUT_BUF_DEPTH, OUT_BUF_HW, OUT_BUF_HW))
    for tr in tqdm(range(NUM_TILE_RC), desc="tile rows"):
        for tc in range(NUM_TILE_RC):
            load_input_tile_from_DRAM_sw(input_fm, conv_in_buf, tr, tc)
            for kg in range(NUM_KG):
                load_layer_params_from_DRAM_sw(weights, biases, conv_wt_buf, conv_bias_buf, kg)
                conv_3x3_sw(conv_in_buf, conv_wt_buf, conv_bias_buf, conv_out_buf)
                store_output_tile_to_DRAM_sw(conv_out_buf, output_fm, tr, tc, kg)
    return output_fm

In [8]:
# Run the software convolution on the CPU (takes several minutes)
print("Starting software convolution...")
output_sw = np.zeros((N, O, O), dtype=np.float64)
sw_start_time = time.time()
tiled_conv_sw(x_r, w_r, b_r, output_sw)
sw_end_time = time.time()
sw_ms = (sw_end_time - sw_start_time) * 1e3
print(f"Software Execution time: {sw_ms / 1e3:.4f} seconds")

Starting software convolution...


tile rows: 100%|██████████| 16/16 [19:07<00:00, 71.73s/it]

Software Execution time: 1147.7484 seconds


In [9]:
# Verify results: hardware output vs software output
hw_q = np.array(output_fm).astype(np.int64)
sw_q = np.floor(output_sw * SCALE).astype(np.int64)
sw_q = ((sw_q + 128) % 256) - 128                      # 8-bit wrap, like the hardware
match = float(np.mean(hw_q == sw_q))
mse_sw = float(np.mean((hw_q / SCALE - output_sw) ** 2))
print(f"HW equals SW on {match * 100:.2f}% of values, MSE (HW vs SW) = {mse_sw:.6f}")
if golden is not None:
    print(f"MSE (HW vs golden file) = {float(np.mean((hw_q / SCALE - golden) ** 2)):.6f}")
print("FPGA output matches CPU output! Verification successful." if match == 1.0
      else "FPGA output does not match CPU output. Verification failed.")

# a few nonzero values (raw 8-bit integers)
nz = np.argwhere(sw_q > 0)
print("filter,row,col | HW | SW")
for f, r, c in nz[::max(1, len(nz) // 8)][:8]:
    print(f, r, c, "|", hw_q[f, r, c], "|", sw_q[f, r, c])

HW equals SW on 100.00% of values, MSE (HW vs SW) = 0.000000
MSE (HW vs golden file) = 0.064847
FPGA output matches CPU output! Verification successful.
filter,row,col | HW | SW
2 1 0 | 2 | 2
8 54 86 | 18 | 18
11 30 223 | 3 | 3
16 172 125 | 7 | 7
23 82 208 | 3 | 3
33 222 80 | 4 | 4
41 218 90 | 7 | 7
53 167 135 | 4 | 4


In [10]:
# Results for the report
# The clock in the block design comes from tiled_conv.hwh. The latency (cycles) and the clock period
# Vitis HLS assumed come from the HLS report (csynth.xml), which must be copied next to this notebook
# (it is in <hls project>/<solution>/syn/report/tiled_conv_csynth.xml).
import re
import xml.etree.ElementTree as ET

# clock the IP is connected to in the block design, read from the .hwh
clk_hz = None
for mod in ET.parse(bitstream.replace(".bit", ".hwh")).getroot().iter("MODULE"):
    if mod.get("INSTANCE") == "tiled_conv_0":
        for port in mod.iter("PORT"):
            if port.get("NAME") == "ap_clk" and port.get("CLKFREQUENCY"):
                clk_hz = float(port.get("CLKFREQUENCY"))
if clk_hz is None:
    raise RuntimeError("could not find the ap_clk frequency in the .hwh")
hwh_mhz = clk_hz / 1e6

# latency (cycles) and target clock period (ns) from the Vitis HLS report
rpt = "tiled_conv_csynth.xml" if os.path.exists("tiled_conv_csynth.xml") else sorted(glob.glob("*csynth*.xml"))[0]
txt = open(rpt).read()
def rpt_value(tag):
    return float(re.search(rf"<{tag}>\s*([0-9.]+)\s*</{tag}>", txt).group(1))
cyc_best, cyc_worst = int(rpt_value("Best-caseLatency")), int(rpt_value("Worst-caseLatency"))
period_ns = rpt_value("TargetClockPeriod")
hls_ms = cyc_worst * period_ns * 1e-6                  # what Vitis HLS predicts
run_mhz = float(Clocks.fclk0_mhz)                      # clock the PL actually ran at during the timing
hw_cycles = hw_ms * 1e-3 * run_mhz * 1e6

print(f"Clock in .hwh      : {hwh_mhz:.1f} MHz   (PL clock during the run: {run_mhz:.1f} MHz, HLS target {1e3 / period_ns:.1f} MHz)")
print(f"Vitis HLS report   : {rpt}")
print(f"Vitis HLS estimate : {cyc_best:,} to {cyc_worst:,} cycles x {period_ns:.1f} ns = {hls_ms:.2f} ms")
print(f"On-board (PL)      : {hw_ms:.2f} ms = {hw_cycles:,.0f} cycles at {run_mhz:.0f} MHz  ({hw_ms / hls_ms:.2f}x the estimate)")
print(f"CPU (PS)           : {sw_ms:,.0f} ms")
print(f"Speed-up PL vs PS  : {sw_ms / hw_ms:,.0f}x")

Clock in .hwh      : 100.0 MHz   (PL clock during the run: 100.0 MHz, HLS target 100.0 MHz)
Vitis HLS report   : tiled_conv_csynth.xml
Vitis HLS estimate : 7,384,065 to 7,384,065 cycles x 10.0 ns = 73.84 ms
On-board (PL)      : 93.18 ms = 9,317,509 cycles at 100 MHz  (1.26x the estimate)
CPU (PS)           : 1,147,748 ms
Speed-up PL vs PS  : 12,318x


In [11]:
# Verify results
hw_q = np.array(output_fm).astype(np.int64)
sw_q = np.floor(output_sw * SCALE).astype(np.int64)
sw_q = ((sw_q + 128) % 256) - 128                      # 8-bit wrap, like the hardware
match = float(np.mean(hw_q == sw_q))

if match == 1.0:
    print("FPGA output matches CPU output! Verification successful.")
else:
    print("FPGA output does not match CPU output. Verification failed.")
print(f"Matching values: {match * 100:.2f}%")
if golden is not None:
    print(f"MSE (HW vs golden file) = {float(np.mean((hw_q / SCALE - golden) ** 2)):.6f}")

FPGA output matches CPU output! Verification successful.
Matching values: 100.00%
MSE (HW vs golden file) = 0.064847


In [12]:
overlay.free()